# Lesson 7.3 — Advanced Security Architecture: Cross-Account Access & SCP Design

**Level: Advanced** | **Exam:** SAP-C02 Domain 1, Tasks 1.2 & 1.4

## Enterprise Security Architecture

### Cross-Account Access Patterns

**Pattern 1: Role Delegation**
```
User in Account A → sts:AssumeRole → Role in Account B
Requires: Role's trust policy allows Account A
Use: Most common cross-account access pattern
```

**Pattern 2: Resource-Based Policy**
```
S3/KMS/Lambda in Account A → allows Account B principal directly
Requires: Resource policy grants the cross-account access
Use: When you want to share resources without role assumption
```

### SCP Design (Denylist vs Allowlist)
- **Denylist (recommended)**: Allow everything except specific denied actions
  - Easier to maintain; new services work automatically
  - Add denies for specific restrictions
  
- **Allowlist (restrictive)**: Block everything except specific allowed services
  - Very secure; requires maintenance as you adopt new services

### Common Production SCPs
```json
1. Prevent disabling CloudTrail/Config (compliance)
2. Restrict to approved regions
3. Prevent creating IAM users (use Identity Center)
4. Require S3 encryption
5. Prevent leaving the organization
6. Require MFA for sensitive IAM actions
```

### Centralized Security Services
Configure these services with **Delegated Administrator**:
- **GuardDuty**: Findings aggregated from all member accounts
- **Security Hub**: All security findings in one place
- **IAM Access Analyzer**: Organization-scope external access detection
- **AWS Config**: Aggregator for all account Config data

In [ ]:
import boto3, json
account_id = boto3.client('sts').get_caller_identity()['Account']
print(f'Account: {account_id}')

In [ ]:
# Analyze cross-account trust relationships
import boto3, json

iam = boto3.client('iam')
account_id = boto3.client('sts').get_caller_identity()['Account']

print(f"Scanning account: {account_id}")
print("Cross-Account Role Trust Relationships:")
print()

paginator = iam.get_paginator('list_roles')
for page in paginator.paginate():
    for role in page['Roles']:
        trust = role['AssumeRolePolicyDocument']
        for stmt in trust.get('Statement', []):
            principal = stmt.get('Principal', {})
            aws_p = principal.get('AWS', []) if isinstance(principal, dict) else []
            if isinstance(aws_p, str):
                aws_p = [aws_p]
            for p in aws_p:
                if account_id not in p:
                    conditions = list(stmt.get('Condition', {}).keys())
                    print(f"  Role: {role['RoleName']:40}")
                    print(f"    Trusts: {p}")
                    print(f"    Conditions: {conditions or 'NONE'}")
                    print()

## Exercise

In [ ]:
# TODO: Design SCPs for a financial services company with these requirements:
# 1. All resources must be in us-east-1 or us-west-2 only (data residency)
# 2. Root account must never be used (all access via IAM Identity Center)
# 3. CloudTrail must never be disabled or deleted  
# 4. No new IAM users can be created (only IAM roles)
# 5. S3 buckets must block all public access

scps = {
    "region_restriction": {
        "Version": "2012-10-17",
        "Statement": [
            {
                "Sid": "DenyUnapprovedRegions",
                # TODO: Deny all actions in non-approved regions
                # Hint: use Condition StringNotEquals aws:RequestedRegion
                # Remember to exempt global services (IAM, STS, Route53, CloudFront)
            }
        ]
    },
    "prevent_iam_users": {
        "Version": "2012-10-17",
        "Statement": [
            # TODO: Deny iam:CreateUser
        ]
    }
    # TODO: Add cloudtrail_protection and s3_public_block SCPs
}

print(json.dumps(scps, indent=2))

## Summary
- **Cross-account role delegation**: Trust policy + permission policy; use ExternalId for third-party
- **SCP denylist** is more practical — allow all, deny specific dangerous actions
- **Delegated Administrator** for GuardDuty, Security Hub, Config = centralized security management  
- **Permission Boundaries** = max permissions cap; enable developer self-service without escalation risk
- Key SCPs for every org: region restriction, CloudTrail protection, no IAM users, no public S3

---
**Next:** [Lesson 7.4 — Cost Optimization: Cost Explorer, Savings Plans & Tagging](lesson-7.4-cost-optimization.ipynb)